# H-07: SLA Breach Rate Estimation

**Question:** Is the SLA breach rate below the allowed target (default 5%) for each sub-fault?

| | |
|---|---|
| **H₀ (Null):** | The true SLA breach rate is at or above the allowed target. |
| **Hₐ (Alternative):** | The true SLA breach rate is below the allowed target. |
| **Certification rule:** | Per-sub-fault breach rate testing, rolled up to category and overall. |
| **Metrics:** | time_to_detect, time_to_mitigate |
| **Methods:** | Exact binomial test + Wilson confidence interval |

### Per-Sub-Fault Verdict Logic
| Condition | Verdict |
|-----------|--------|
| Binomial p < α (breach rate provably below target) | ✅ PASS |
| CI lower bound > target (breach rate clearly above target) | ❌ FAIL |
| Cannot reject H₀ but observed rate below target | ⚠️ INCONCLUSIVE |
| No SLA threshold in ground truth | ❓ NO_SLA_DEFINED |

### Category Rollup
| Condition | Category Verdict |
|-----------|------------------|
| All sub-faults PASS | PASS |
| Any sub-fault FAIL | FAIL |
| Any NO_SLA_DEFINED (none FAIL) | INCOMPLETE |
| Any INCONCLUSIVE (none FAIL/INCOMPLETE) | INCONCLUSIVE |

### Key Difference from H-06
H-07 uses **ALL runs** (not just detected). Non-detected runs are represented as `float('inf')`,
so they automatically count as breaches against any finite SLA threshold.


In [1]:
import sys, os
import json
import yaml
import numpy as np
from pathlib import Path
from collections import defaultdict

project_root = Path(os.getcwd()).parent.parent.parent
sys.path.insert(0, str(project_root))

data_root = project_root / "hypothesis_framework" / "data" / "input"
gt_root = project_root / "hypothesis_framework" / "data" / "groundtruth" / "kubernetes"

def load_runs(category_dir):
    runs = []
    for f in sorted(category_dir.glob("*.json")):
        run = json.loads(f.read_text(encoding="utf-8"))
        runs.append(run)
    return runs

def build_subfault_data_all(runs, metric_name):
    """All runs. Non-detected get inf (counts as breach)."""
    grouped = defaultdict(list)
    for run in runs:
        q = run["quantitative"]
        val = q.get(metric_name)
        if val is not None:
            grouped[run["fault_name"]].append(float(val))
        else:
            grouped[run["fault_name"]].append(float("inf"))
    return dict(grouped)

def load_sla_thresholds(gt_dir, metric_key):
    """Load SLA thresholds from ground truth YAML files."""
    thresholds = {}
    for fault_dir in sorted(gt_dir.iterdir()):
        gt_file = fault_dir / "ground_truth.yaml"
        if not gt_file.exists():
            continue
        with open(gt_file, "r", encoding="utf-8") as f:
            gt = yaml.safe_load(f)
        sla = gt.get("ground_truth", {}).get("sla", {})
        metric_sla = sla.get(metric_key, {})
        if "threshold" in metric_sla:
            thresholds[fault_dir.name] = float(metric_sla["threshold"])
    return thresholds

# Load runs
categories = {}
for cat_name in ["application_fault", "network_fault", "resource_fault"]:
    cat_dir = data_root / cat_name
    if cat_dir.exists():
        categories[cat_name] = load_runs(cat_dir)

# Load SLA thresholds
ttd_sla = load_sla_thresholds(gt_root, "time_to_detect")
ttm_sla = load_sla_thresholds(gt_root, "time_to_mitigate")

print("=== Data Loaded (ALL runs, non-detected \u2192 inf) ===")
for cat, runs in categories.items():
    total = len(runs)
    detected = sum(1 for r in runs if r['quantitative'].get('fault_detected') == 'Yes')
    faults = sorted(set(r['fault_name'] for r in runs))
    print(f"  {cat}: {total} runs, {detected} detected ({detected/total:.0%})")
    for fn in faults:
        fn_total = sum(1 for r in runs if r['fault_name'] == fn)
        fn_det = sum(1 for r in runs if r['fault_name'] == fn and r['quantitative'].get('fault_detected') == 'Yes')
        sla_ttd = ttd_sla.get(fn, '\u2014')
        sla_ttm = ttm_sla.get(fn, '\u2014')
        print(f"    {fn}: {fn_total} total, {fn_det} detected  (SLA: TTD={sla_ttd}s, TTM={sla_ttm}s)")


=== Data Loaded (ALL runs, non-detected → inf) ===
  application_fault: 60 runs, 51 detected (85%)
    container-kill: 30 total, 28 detected  (SLA: TTD=—s, TTM=—s)
    pod-delete: 30 total, 23 detected  (SLA: TTD=60.0s, TTM=120.0s)
  network_fault: 90 runs, 39 detected (43%)
    pod-dns-error: 30 total, 13 detected  (SLA: TTD=60.0s, TTM=180.0s)
    pod-network-corruption: 30 total, 14 detected  (SLA: TTD=240.0s, TTM=420.0s)
    pod-network-loss: 30 total, 12 detected  (SLA: TTD=180.0s, TTM=360.0s)
  resource_fault: 90 runs, 70 detected (78%)
    disk-fill: 30 total, 25 detected  (SLA: TTD=60.0s, TTM=180.0s)
    pod-cpu-hog: 30 total, 25 detected  (SLA: TTD=120.0s, TTM=300.0s)
    pod-memory-hog: 30 total, 20 detected  (SLA: TTD=90.0s, TTM=240.0s)


---
## Step 1: SLA Thresholds from Ground Truth

Shows per-sub-fault SLA thresholds loaded from the ground truth YAML files.


In [2]:
print("SLA Thresholds (from ground truth)")
print("=" * 70)
print(f"{'Fault':30s} {'TTD (s)':>10s} {'TTM (s)':>10s}")
print("-" * 70)

all_faults = sorted(set(list(ttd_sla.keys()) + list(ttm_sla.keys())))
for fn in all_faults:
    ttd = f"{ttd_sla[fn]:.0f}" if fn in ttd_sla else "\u2014"
    ttm = f"{ttm_sla[fn]:.0f}" if fn in ttm_sla else "\u2014"
    in_data = any(fn in [r['fault_name'] for r in runs] for runs in categories.values())
    marker = "  \u2190 in dataset" if in_data else ""
    print(f"  {fn:28s} {ttd:>10s} {ttm:>10s}{marker}")

# Check for faults in data without SLA
data_faults = set()
for runs in categories.values():
    for r in runs:
        data_faults.add(r['fault_name'])
missing_sla = data_faults - set(ttd_sla.keys())
if missing_sla:
    print(f"\n\u26a0\ufe0f Faults in dataset WITHOUT ground truth SLA: {sorted(missing_sla)}")
    print("  These will be marked as NO_SLA_DEFINED in the test results.")

print(f"\nTarget breach rate: 5% (default)")


SLA Thresholds (from ground truth)
Fault                             TTD (s)    TTM (s)
----------------------------------------------------------------------
  disk-fill                            60        180  ← in dataset
  node-restart                         30        300
  pod-autoscaler                      120        180
  pod-cpu-hog                         120        300  ← in dataset
  pod-delete                           60        120  ← in dataset
  pod-dns-error                        60        180  ← in dataset
  pod-memory-hog                       90        240  ← in dataset
  pod-network-corruption              240        420  ← in dataset
  pod-network-loss                    180        360  ← in dataset
  pod-network-rate-limit              180        360

⚠️ Faults in dataset WITHOUT ground truth SLA: ['container-kill']
  These will be marked as NO_SLA_DEFINED in the test results.

Target breach rate: 5% (default)


---
## Step 2: Run H-07 — time_to_detect

Tests each sub-fault’s breach rate against the target (5%) for time-to-detect.
All runs included — non-detected runs count as `inf` (automatic breach).


In [3]:
from hypothesis_framework.scripts.hypothesis_tests.h07_sla_breach_rate import run_breach_rate_test

ttd_data = {}
for cat, runs in categories.items():
    ttd_data[cat] = build_subfault_data_all(runs, "time_to_detect")

h07_ttd = run_breach_rate_test(
    ttd_data,
    sla_thresholds=ttd_sla,
    target_rate=0.05,
    metric_name="time_to_detect",
)

print(f"H-07: {h07_ttd.hypothesis_name}")
print(f"Metric: {h07_ttd.metric_name}  |  Target breach rate: {h07_ttd.target_rate:.0%}")
print(f"Overall: {h07_ttd.overall_assessment}")
print("=" * 100)

for c in h07_ttd.per_category:
    v_icon = "\u2705" if c.verdict == "PASS" else "\u274c" if c.verdict == "FAIL" else "\u26a0\ufe0f"
    print(f"\n  {c.category} (n={c.n}, {c.n_sub_faults} sub-faults):")
    print(f"    Verdict: {v_icon} {c.verdict}  "
          f"(pass={c.n_passed}, fail={c.n_failed}, inc={c.n_inconclusive}, no_sla={c.n_no_sla})")
    if c.worst_sub_fault:
        print(f"    Worst sub-fault: {c.worst_sub_fault}")
    for sf in c.sub_faults:
        sf_icon = "\u2705" if sf.verdict == "PASS" else "\u274c" if sf.verdict == "FAIL" else "\u2753" if sf.verdict == "NO_SLA_DEFINED" else "\u26a0\ufe0f"
        sla_str = f"SLA={sf.sla_threshold:.0f}s" if sf.sla_threshold is not None else "no SLA"
        bp = f"{sf.binomial_p:.6f}" if sf.binomial_p is not None else "\u2014"
        rate_str = f"{sf.observed_rate:.2%}" if sf.trials > 0 and sf.verdict != "NO_SLA_DEFINED" else "\u2014"
        ci_lo = f"{sf.ci_lower:.4f}" if sf.binomial_p is not None else "\u2014"
        ci_hi = f"{sf.ci_upper:.4f}" if sf.binomial_p is not None else "\u2014"
        breach_str = f"{sf.breaches}/{sf.trials}" if sf.binomial_p is not None else "\u2014"
        print(f"      {sf.fault_name:25s}: {breach_str:>8s}  rate={rate_str:>8s}  {sla_str:12s}  "
              f"binom_p={bp:>10s}  CI=[{ci_lo:>8s}, {ci_hi:>8s}]  {sf_icon} {sf.verdict}")

if h07_ttd.warnings:
    print(f"\nWarnings:")
    for w in h07_ttd.warnings:
        print(f"  - {w}")


H-07: SLA Breach Rate Estimation
Metric: time_to_detect  |  Target breach rate: 5%
Overall: breach_rate_exceeds_target

  application_fault (n=60, 2 sub-faults):
    Verdict: ❌ FAIL  (pass=0, fail=1, inc=0, no_sla=1)
    Worst sub-fault: pod-delete
      container-kill           :        —  rate=       —  no SLA        binom_p=         —  CI=[       —,        —]  ❓ NO_SLA_DEFINED
      pod-delete               :    30/30  rate= 100.00%  SLA=60s       binom_p=  1.000000  CI=[  0.8865,   1.0000]  ❌ FAIL

  network_fault (n=90, 3 sub-faults):
    Verdict: ❌ FAIL  (pass=0, fail=3, inc=0, no_sla=0)
    Worst sub-fault: pod-dns-error
      pod-dns-error            :    23/30  rate=  76.67%  SLA=60s       binom_p=  1.000000  CI=[  0.5907,   0.8821]  ❌ FAIL
      pod-network-corruption   :    21/30  rate=  70.00%  SLA=240s      binom_p=  1.000000  CI=[  0.5212,   0.8334]  ❌ FAIL
      pod-network-loss         :    21/30  rate=  70.00%  SLA=180s      binom_p=  1.000000  CI=[  0.5212,   0.8334] 

---
## Step 3: Run H-07 — time_to_mitigate

Tests each sub-fault’s breach rate against the target (5%) for time-to-mitigate.


In [4]:
ttm_data = {}
for cat, runs in categories.items():
    ttm_data[cat] = build_subfault_data_all(runs, "time_to_mitigate")

h07_ttm = run_breach_rate_test(
    ttm_data,
    sla_thresholds=ttm_sla,
    target_rate=0.05,
    metric_name="time_to_mitigate",
)

print(f"H-07: {h07_ttm.metric_name}  |  {h07_ttm.overall_assessment}")
print("=" * 100)
for c in h07_ttm.per_category:
    v_icon = '\u2705' if c.verdict == 'PASS' else '\u274c' if c.verdict == 'FAIL' else '\u26a0\ufe0f'
    print(f"  {c.category:20s}: {v_icon} {c.verdict:14s}  "
          f"(pass={c.n_passed}, fail={c.n_failed}, inc={c.n_inconclusive}, no_sla={c.n_no_sla})")
    for sf in c.sub_faults:
        sf_icon = '\u2705' if sf.verdict == 'PASS' else '\u274c' if sf.verdict == 'FAIL' else '\u2753' if sf.verdict == 'NO_SLA_DEFINED' else '\u26a0\ufe0f'
        sla_str = f"SLA={sf.sla_threshold:.0f}s" if sf.sla_threshold is not None else "no SLA"
        rate_str = f"{sf.observed_rate:.2%}" if sf.trials > 0 and sf.verdict != "NO_SLA_DEFINED" else "\u2014"
        breach_str = f"{sf.breaches}/{sf.trials}" if sf.binomial_p is not None else "\u2014"
        print(f"    {sf.fault_name:25s}: {breach_str:>8s}  rate={rate_str:>8s}  {sla_str:12s}  {sf_icon} {sf.verdict}")

if h07_ttm.warnings:
    print(f"\nWarnings: {h07_ttm.warnings}")


H-07: time_to_mitigate  |  breach_rate_exceeds_target
  application_fault   : ❌ FAIL            (pass=0, fail=1, inc=0, no_sla=1)
    container-kill           :        —  rate=       —  no SLA        ❓ NO_SLA_DEFINED
    pod-delete               :    30/30  rate= 100.00%  SLA=120s      ❌ FAIL
  network_fault       : ❌ FAIL            (pass=0, fail=3, inc=0, no_sla=0)
    pod-dns-error            :    29/30  rate=  96.67%  SLA=180s      ❌ FAIL
    pod-network-corruption   :    24/30  rate=  80.00%  SLA=420s      ❌ FAIL
    pod-network-loss         :    25/30  rate=  83.33%  SLA=360s      ❌ FAIL
  resource_fault      : ❌ FAIL            (pass=0, fail=3, inc=0, no_sla=0)
    disk-fill                :    29/30  rate=  96.67%  SLA=180s      ❌ FAIL
    pod-cpu-hog              :    28/30  rate=  93.33%  SLA=300s      ❌ FAIL
    pod-memory-hog           :    30/30  rate= 100.00%  SLA=240s      ❌ FAIL

Warnings: ['application_fault/container-kill: no SLA threshold defined — skipping breach te

---
## Step 4: Summary & Interpretation


In [5]:
print("H-07 SLA Breach Rate Estimation \u2014 Summary")
print("=" * 90)

results = {
    "time_to_detect": h07_ttd,
    "time_to_mitigate": h07_ttm,
}

for metric, r in results.items():
    overall_icon = '\u2705' if r.overall_assessment == 'breach_rate_certified' else '\u274c' if r.overall_assessment == 'breach_rate_exceeds_target' else '\u26a0\ufe0f'
    print(f"\n  {metric}: {overall_icon} {r.overall_assessment}")
    for c in r.per_category:
        v_icon = '\u2705' if c.verdict == 'PASS' else '\u274c' if c.verdict == 'FAIL' else '\u26a0\ufe0f'
        print(f"    {c.category:20s}: {v_icon} {c.verdict:14s}  "
              f"({c.n_passed}P/{c.n_failed}F/{c.n_inconclusive}I/{c.n_no_sla}U)")

print("\n" + "=" * 90)
print("\nInterpretation:")
print("  - PASS: Exact binomial test confirms breach rate < target (5%).")
print("  - FAIL: Wilson CI lower bound exceeds target \u2014 breach rate clearly too high.")
print("  - INCONCLUSIVE: Cannot reject H\u2080 but observed rate is below target \u2014 more data needed.")
print("  - INCOMPLETE: Some sub-faults lack ground truth SLA thresholds.")
print("  - NO_SLA_DEFINED: Fault type has no SLA in ground truth (e.g., container-kill).")
print("  - P/F/I/U = Pass/Fail/Inconclusive/Unassessed sub-faults.")
print("  - Non-detected runs are included as inf \u2192 automatic SLA breach.")


H-07 SLA Breach Rate Estimation — Summary

  time_to_detect: ❌ breach_rate_exceeds_target
    application_fault   : ❌ FAIL            (0P/1F/0I/1U)
    network_fault       : ❌ FAIL            (0P/3F/0I/0U)
    resource_fault      : ❌ FAIL            (0P/3F/0I/0U)

  time_to_mitigate: ❌ breach_rate_exceeds_target
    application_fault   : ❌ FAIL            (0P/1F/0I/1U)
    network_fault       : ❌ FAIL            (0P/3F/0I/0U)
    resource_fault      : ❌ FAIL            (0P/3F/0I/0U)


Interpretation:
  - PASS: Exact binomial test confirms breach rate < target (5%).
  - FAIL: Wilson CI lower bound exceeds target — breach rate clearly too high.
  - INCONCLUSIVE: Cannot reject H₀ but observed rate is below target — more data needed.
  - INCOMPLETE: Some sub-faults lack ground truth SLA thresholds.
  - NO_SLA_DEFINED: Fault type has no SLA in ground truth (e.g., container-kill).
  - P/F/I/U = Pass/Fail/Inconclusive/Unassessed sub-faults.
  - Non-detected runs are included as inf → automati

---
## Step 5: JSON Output


In [6]:
output = {
    "hypothesis_id": "H-07",
    "hypothesis_name": "SLA Breach Rate Estimation",
    "null_hypothesis": h07_ttd.null_hypothesis,
    "alt_hypothesis": h07_ttd.alt_hypothesis,
    "certification_rule": "Per-sub-fault breach rate testing with exact binomial + Wilson CI.",
    "hypothesis_metadata": {
        "methods": [
            "Exact binomial test",
            "Wilson confidence interval",
        ],
        "alpha": 0.05,
        "target_rate": 0.05,
        "data_mode": "all_runs (non-detected = inf)",
        "sla_source": "data/groundtruth/kubernetes/*/ground_truth.yaml",
        "sla_thresholds": {
            "time_to_detect": ttd_sla,
            "time_to_mitigate": ttm_sla,
        },
        "categories": list(categories.keys()),
    },
    "results": {},
}

for metric, r in results.items():
    output["results"][metric] = {
        "overall_assessment": r.overall_assessment,
        "per_category": [
            {
                "category": c.category,
                "n": c.n,
                "n_sub_faults": c.n_sub_faults,
                "verdict": c.verdict,
                "n_passed": c.n_passed,
                "n_failed": c.n_failed,
                "n_inconclusive": c.n_inconclusive,
                "n_no_sla": c.n_no_sla,
                "worst_sub_fault": c.worst_sub_fault,
                "sub_faults": [
                    {
                        "fault_name": sf.fault_name,
                        "breaches": sf.breaches,
                        "trials": sf.trials,
                        "observed_rate": sf.observed_rate,
                        "target_rate": sf.target_rate,
                        "sla_threshold": sf.sla_threshold,
                        "binomial_p": sf.binomial_p,
                        "ci_lower": sf.ci_lower,
                        "ci_upper": sf.ci_upper,
                        "verdict": sf.verdict,
                    }
                    for sf in c.sub_faults
                ],
            }
            for c in r.per_category
        ],
        "warnings": r.warnings,
    }

print(json.dumps(output, indent=2, default=str, ensure_ascii=False))


{
  "hypothesis_id": "H-07",
  "hypothesis_name": "SLA Breach Rate Estimation",
  "null_hypothesis": "The true SLA breach rate is at or above the allowed target.",
  "alt_hypothesis": "The true SLA breach rate is below the allowed target.",
  "certification_rule": "Per-sub-fault breach rate testing with exact binomial + Wilson CI.",
  "hypothesis_metadata": {
    "methods": [
      "Exact binomial test",
      "Wilson confidence interval"
    ],
    "alpha": 0.05,
    "target_rate": 0.05,
    "data_mode": "all_runs (non-detected = inf)",
    "sla_source": "data/groundtruth/kubernetes/*/ground_truth.yaml",
    "sla_thresholds": {
      "time_to_detect": {
        "disk-fill": 60.0,
        "node-restart": 30.0,
        "pod-autoscaler": 120.0,
        "pod-cpu-hog": 120.0,
        "pod-delete": 60.0,
        "pod-dns-error": 60.0,
        "pod-memory-hog": 90.0,
        "pod-network-corruption": 240.0,
        "pod-network-loss": 180.0,
        "pod-network-rate-limit": 180.0
      },
 